# Time-Use Profiles: Archetype-Based Assignment

Run the shared archetype assignment workflow for weekday and weekend, then save the reference draw and multi-draw outputs.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.timeuse_profiles import prepare_timeuse_data, run_archetype_assignment_workflow


In [ ]:
DAY_TYPES = ["weekday", "weekend"]
LIMIT_TO_SELECTED_DAS = True
N_SELECTED_DAS = 50
OUTPUT_TAG = "archetype_da50" if LIMIT_TO_SELECTED_DAS else "archetype"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "timeuse_profiles"


In [ ]:
prepared = prepare_timeuse_data(
    limit_to_selected_das=LIMIT_TO_SELECTED_DAS,
    n_selected_das=N_SELECTED_DAS,
)

results_by_day = {}
for day_type in DAY_TYPES:
    result = run_archetype_assignment_workflow(prepared, day_type=day_type)
    results_by_day[day_type] = result

    result["reference_assignments"].to_csv(OUT_DIR / f"synthetic_to_tus_donors_{day_type}_{OUTPUT_TAG}.csv", index=False)
    result["reference_district_profiles"].drop(columns=["draw_id", "day_type"], errors="ignore").to_csv(
        OUT_DIR / f"district_hourly_profiles_{day_type}_{OUTPUT_TAG}.csv", index=False
    )
    result["donor_draws"].to_csv(OUT_DIR / f"synthetic_to_tus_donors_{day_type}_{OUTPUT_TAG}_draws.csv", index=False)
    result["district_draws"].to_csv(OUT_DIR / f"district_hourly_profiles_{day_type}_{OUTPUT_TAG}_draws.csv", index=False)
    result["donor_draw_summary"].to_csv(OUT_DIR / f"synthetic_to_tus_donors_{day_type}_{OUTPUT_TAG}_draw_summary.csv", index=False)
    result["district_uncertainty"].to_csv(OUT_DIR / f"district_hourly_profiles_{day_type}_{OUTPUT_TAG}_uncertainty.csv", index=False)
    result["hourly_uncertainty"].to_csv(OUT_DIR / f"mean_hourly_profiles_{day_type}_{OUTPUT_TAG}_uncertainty.csv", index=False)
    result["respondent_archetypes"].to_csv(OUT_DIR / f"tus_respondent_archetypes_{day_type}_{OUTPUT_TAG}.csv", index=False)
    result["centroid_profiles"].to_csv(OUT_DIR / f"tus_archetype_centroids_{day_type}_{OUTPUT_TAG}.csv", index=False)
    result["fit_summary"].to_csv(OUT_DIR / f"tus_archetype_fit_summary_{day_type}_{OUTPUT_TAG}.csv", index=False)

    display(result["fit_summary"])
    display(result["donor_draw_summary"])
